In [1]:
import numpy as np
import pandas as pd
import random
import uuid
import time
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy.spatial.distance import mahalanobis

# Diffie-Hellman params
q = 7919
alpha = 2

def modular_pow(base, exponent, modulus):
    result = 1
    base = base % modulus
    while exponent > 0:
        if exponent % 2:
            result = (result * base) % modulus
        exponent = exponent >> 1
        base = (base * base) % modulus
    return result

class DataCapsule:
    def __init__(self, features, label, source_id, pca_model=None, label_counts=None):
        self.id = str(uuid.uuid4())
        self.features = features
        self.label = label
        self.source_id = source_id
        self.timestamp = time.time()
        self.confidence = self.calculate_confidence(pca_model)
        self.rare = self.calculate_rarity(features, label, pca_model, label_counts)

    def calculate_confidence(self, pca_model):
        if pca_model is None:
            return 1.0

        if not hasattr(pca_model, 'transformed_train_'):
            pca_model.transformed_train_ = pca_model.transform(pca_model._fit_X)

        transformed = pca_model.transform([self.features])
        centroid = np.mean(pca_model.transformed_train_, axis=0)
        distance = np.linalg.norm(transformed[0] - centroid)
        max_distance = np.max([np.linalg.norm(x - centroid) for x in pca_model.transformed_train_])

        confidence = 1 - (distance / max_distance) if max_distance != 0 else 1.0
        return float(np.clip(confidence, 0, 1))

    def calculate_rarity(self, features, label, pca_model, label_counts):
        rare_label = label_counts.get(label, 0) < 0.1 * sum(label_counts.values())
        low_conf = self.confidence < 0.6
        if pca_model:
            inv_cov = np.linalg.inv(np.cov(pca_model.transformed_train_.T))
            transformed = pca_model.transform([features])
            mahal_dist = mahalanobis(transformed[0], np.mean(pca_model.transformed_train_, axis=0), inv_cov)
            rare_dist = mahal_dist > 2.5
        else:
            rare_dist = False
        return rare_label or low_conf or rare_dist

class ModelCapsule:
    def __init__(self, model_id, model_type, accuracy, capsule_ids, high_conf_capsules, rare_capsules, weights, intercept):
        self.model_id = model_id
        self.model_type = model_type
        self.accuracy = accuracy
        self.capsule_ids = capsule_ids
        self.high_conf_capsules = high_conf_capsules
        self.rare_capsules = rare_capsules
        self.weights = weights
        self.intercept = intercept
        self.created_at = time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime())
        self.model_summary = {}

class ClientContext:
    def __init__(self):
        self.seen_capsules = set()
        self.rare_coverage = set()
        self.accuracy_history = []

    def update_context(self, model_capsule):
        self.seen_capsules.update(model_capsule.capsule_ids)
        self.rare_coverage.update(model_capsule.rare_capsules)
        self.accuracy_history.append(model_capsule.accuracy)

    def score_capsule(self, model_capsule):
        novelty = len(set(model_capsule.capsule_ids) - self.seen_capsules)
        rare_value = len(set(model_capsule.rare_capsules) - self.rare_coverage)
        accuracy_gain = model_capsule.accuracy - np.mean(self.accuracy_history[-3:] or [0])
        score = 0.4 * novelty + 0.4 * rare_value + 0.2 * accuracy_gain
        return score

class Node:
    def __init__(self, id, data, labels):
        self.id = id
        self.private_key = random.randint(2, q - 2)
        self.public_key = modular_pow(alpha, self.private_key, q)
        self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(data, labels, test_size=0.2, random_state=42)

        self.model_weights = None
        self.intercept = 0

        self.ewc_lambda = 100
        self.ewc_importance = None
        self.optimal_weights = None

        self.pca_model = PCA(n_components=min(self.X_train.shape[1], 5))
        self.pca_model._fit_X = self.X_train
        self.pca_model.fit(self.X_train)
        self.label_counts = dict(pd.Series(self.y_train).value_counts())
        self.data_capsules = self._generate_data_capsules()
        self.context = ClientContext()
        self.model_capsules = []

    def _generate_data_capsules(self):
        return [
            DataCapsule(features=self.X_train[i], label=self.y_train[i], source_id=self.id,
                        pca_model=self.pca_model, label_counts=self.label_counts)
            for i in range(len(self.X_train))
        ]

    def compute_shared_key(self, other_public_key):
        return modular_pow(other_public_key, self.private_key, q)

    def encrypt_model(self, weights, intercept, shared_key):
        enc_weights = [w + shared_key for w in weights]
        enc_intercept = intercept + shared_key
        return enc_weights, enc_intercept

    def decrypt_model(self, enc_weights, enc_intercept, shared_key):
        self.model_weights = np.array([w - shared_key for w in enc_weights])
        self.intercept = enc_intercept - shared_key

    def estimate_importance(self):
        clf = SGDClassifier(loss='log_loss', max_iter=1, learning_rate='constant', eta0=0.01, random_state=42)
        clf.partial_fit(self.X_train, self.y_train, classes=[0, 1])
        grads = clf.coef_[0]
        self.ewc_importance = grads ** 2
        self.optimal_weights = np.copy(self.model_weights)

    def train_model(self):
        clf = SGDClassifier(loss='log_loss', max_iter=1000, learning_rate='constant', eta0=0.01, random_state=42)
        if self.model_weights is not None:
            clf.coef_ = self.model_weights.reshape(1, -1)
            clf.intercept_ = np.array([self.intercept])
            clf.classes_ = np.array([0, 1])
        clf.partial_fit(self.X_train, self.y_train, classes=[0, 1])
        updated_weights = clf.coef_[0]
        updated_intercept = clf.intercept_[0]

        if self.ewc_importance is not None and self.optimal_weights is not None:
            penalty = self.ewc_lambda * self.ewc_importance * (updated_weights - self.optimal_weights) ** 2
            updated_weights -= 0.01 * penalty

        self.model_weights = updated_weights
        self.intercept = updated_intercept

        acc = self.evaluate()
        capsule_ids = [dc.id for dc in self.data_capsules]
        high_conf_capsules = [dc.id for dc in self.data_capsules if dc.confidence >= 0.85]
        rare_capsules = [dc.id for dc in self.data_capsules if dc.rare]
        capsule = ModelCapsule(
            model_id=str(uuid.uuid4()),
            model_type="SGDClassifier",
            accuracy=acc,
            capsule_ids=capsule_ids,
            high_conf_capsules=high_conf_capsules,
            rare_capsules=rare_capsules,
            weights=self.model_weights,
            intercept=self.intercept
        )
        self.model_capsules.append(capsule)
        self.context.update_context(capsule)

    def evaluate(self):
        clf = SGDClassifier()
        clf.coef_ = self.model_weights.reshape(1, -1)
        clf.intercept_ = np.array([self.intercept])
        clf.classes_ = np.array([0, 1])
        preds = clf.predict(self.X_test)
        return accuracy_score(self.y_test, preds)

def load_and_prepare(filepath):
    df = pd.read_csv(filepath)
    df.columns.values[-1] = 'target'
    y = df['target'].values
    X = df.drop(columns=['target']).values
    scaler = StandardScaler()
    X = scaler.fit_transform(X)
    return X, y

def secure_gossip_training_contextual(num_nodes=3, num_rounds=3):
    heart_X, heart_y = load_and_prepare('heart_dataset_1000.csv')
    kidney_X, kidney_y = load_and_prepare('kidney_dataset_1000.csv')
    lung_X, lung_y = load_and_prepare('lung_dataset_1000.csv')

    datasets = [(heart_X, heart_y), (kidney_X, kidney_y), (lung_X, lung_y)]
    nodes = [Node(i, datasets[i][0], datasets[i][1]) for i in range(num_nodes)]

    print(f"\n\U0001f9e0 Initial training on Node 0")
    nodes[0].train_model()
    print(f"Node 0 Initial Accuracy: {nodes[0].model_capsules[-1].accuracy:.4f}")
    nodes[0].estimate_importance()

    for round_num in range(num_rounds):
        print(f"\n\U0001f501 Round {round_num + 1}")
        for sender in nodes:
            best_score = float('-inf')
            best_receiver = None
            for receiver in nodes:
                if receiver.id == sender.id:
                    continue
                score = receiver.context.score_capsule(sender.model_capsules[-1])
                if score > best_score:
                    best_score = score
                    best_receiver = receiver

            if best_receiver:
                print(f"\U0001f4e4 Node {sender.id} pushes capsule to Node {best_receiver.id} (score: {best_score:.2f})")
                shared_key_send = sender.compute_shared_key(best_receiver.public_key)
                shared_key_recv = best_receiver.compute_shared_key(sender.public_key)

                enc_weights, enc_intercept = sender.encrypt_model(sender.model_weights, sender.intercept, shared_key_send)
                best_receiver.decrypt_model(enc_weights, enc_intercept, shared_key_recv)

                best_receiver.train_model()
                if best_receiver.ewc_importance is None:
                    best_receiver.estimate_importance()

                print(f"\U0001f3af Node {best_receiver.id} Accuracy after training: {best_receiver.model_capsules[-1].accuracy:.4f}")

        print(f"\n\U0001f4ca Accuracy after Round {round_num + 1}")
        for node in nodes:
            acc = node.evaluate()
            print(f"Node {node.id} Test Accuracy: {acc:.4f}")

    return nodes

def evaluate_nodes(nodes):
    print("\n\U0001f4ca Final Evaluation Results:")
    for node in nodes:
        acc = node.evaluate()
        print(f"Node {node.id} Test Accuracy: {acc:.4f}")

# Run
nodes = secure_gossip_training_contextual()
evaluate_nodes(nodes)



🧠 Initial training on Node 0
Node 0 Initial Accuracy: 0.8950

🔁 Round 1
📤 Node 0 pushes capsule to Node 1 (score: 566.18)
🎯 Node 1 Accuracy after training: 0.8850
📤 Node 1 pushes capsule to Node 2 (score: 522.18)
🎯 Node 2 Accuracy after training: 0.9000
📤 Node 2 pushes capsule to Node 1 (score: 557.60)
🎯 Node 1 Accuracy after training: 0.8700

📊 Accuracy after Round 1
Node 0 Test Accuracy: 0.8950
Node 1 Test Accuracy: 0.8700
Node 2 Test Accuracy: 0.9000

🔁 Round 2
📤 Node 0 pushes capsule to Node 1 (score: 566.00)
🎯 Node 1 Accuracy after training: 0.8850
📤 Node 1 pushes capsule to Node 0 (score: 522.00)
🎯 Node 0 Accuracy after training: 0.8700
📤 Node 2 pushes capsule to Node 1 (score: 557.60)
🎯 Node 1 Accuracy after training: 0.8700

📊 Accuracy after Round 2
Node 0 Test Accuracy: 0.8700
Node 1 Test Accuracy: 0.8700
Node 2 Test Accuracy: 0.9000

🔁 Round 3
📤 Node 0 pushes capsule to Node 1 (score: 566.00)
🎯 Node 1 Accuracy after training: 0.8900
📤 Node 1 pushes capsule to Node 0 (score: 